# 第 6 章 智能体与记忆（Agent & Memory）

**学习目标**
- 用 v1 的 `create_agent` 快速创建工具智能体
- 理解 Agent 的消息流转（比第 5 章的手写循环多了什么）
- 用 `checkpointer` + `thread_id` 给 Agent 加多轮记忆
- 掌握 `response_format` 让 Agent 输出结构化结果
- 建立"链 vs Agent"的选型判断

> **链 vs Agent**：链是固定的流程（你写好每一步），Agent 是动态的（模型自己决定下一步做什么）。简单流程用链，开放任务用 Agent。

In [ ]:
# 环境准备：加载 .env 并初始化模型（本系列所有章节通用）
import os
from dotenv import load_dotenv

load_dotenv()  # 读取项目根目录下的 .env 文件

from langchain_openai import ChatOpenAI

# 智谱 GLM 提供 OpenAI 兼容接口，直接用 ChatOpenAI 指向智谱的 base_url 即可
llm = ChatOpenAI(
    model="glm-4.5-air",  # 可换成你账号可用的其他模型（改模型名即可，无需改其他代码）
    temperature=0.6,
    api_key=os.getenv("API_KEY_ZHIPU"),
    base_url=os.getenv("API_URL_ZHIPU"),  # 形如 https://open.bigmodel.cn/api/paas/v4/
)

print("模型就绪:", llm.model_name)
# 注意：本章会发起若干次真实 API 调用，会产生少量费用；练习请使用短文本。

## 准备：先定义几个练习工具

本章所有 Agent 共用这 3 个工具（伪数据，不产生真实副作用）：

In [ ]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def calculator(expression: str) -> str:
    """计算一个四则运算表达式，如 '3 * (12 + 8)'。只支持数字和 + - * / ( )。"""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "错误：表达式包含不支持的字符"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"计算出错: {e}"

@tool
def query_order(order_id: str) -> str:
    """根据订单号查询订单状态。订单号形如 SO 开头。"""
    fake_orders = {
        "SO20260901": "已发货，物流显示运输中，预计明天送达",
        "SO20260815": "已签收",
    }
    return fake_orders.get(order_id, f"未找到订单 {order_id}")

@tool
def current_time() -> str:
    """获取当前的日期和时间。涉及'今天/现在'等时间问题时使用。"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculator, query_order, current_time]
print("工具注册完成:", [t.name for t in tools])

## 1. create_agent：三行代码一个 Agent

v1 版 LangChain 中，`create_agent` 从 `langchain.agents` 导入（旧版的 `create_react_agent` 已归入 `langchain_classic`）。它内部就是第 5 章的手写循环 + 更完善的调度（基于 LangGraph 实现）。

返回值是一个可编译执行的图（graph），用 `.invoke({"messages": [...]})` 触发。

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "你是一位电商客服助手。回答问题前先判断是否需要调用工具；"
        "涉及计算务必用计算器工具，不要心算；回复简洁友好。"
    ),
)

result = agent.invoke({"messages": [{"role": "user", "content": "订单 SO20260901 到哪了？预计什么时候能到？"}]})
print(result["messages"][-1].content)

## 2. 观察 Agent 的消息流转

`result["messages"]` 是完整的消息轨迹：用户消息 → （带 tool_calls 的 AI 消息 → ToolMessage）× N 轮 → 最终 AI 回答。看懂这个序列，你就看懂了 Agent。

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "帮我算一下 3.8 折 4999 元的手机多少钱，再看看现在几点了"}]})

for m in result["messages"]:
    role = type(m).__name__
    if role == "AIMessage" and m.tool_calls:
        print(f"{role:12} | 调用工具: {[tc['name'] for tc in m.tool_calls]} 参数: {[tc['args'] for tc in m.tool_calls]}")
    elif role == "ToolMessage":
        print(f"{role:12} | 结果: {m.content}")
    else:
        print(f"{role:12} | {str(m.content)[:60]}")

## 3. 给 Agent 加记忆：checkpointer

Agent 默认每次 `invoke` 都是全新的（无状态）。给它一个 `checkpointer`（检查点存储器），并用 `thread_id` 标识会话，Agent 就能记住同一个会话里的所有历史——**这就是"多轮对话记忆"的全部实现**。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

chat_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是电商客服助手，自然地记住用户在会话中告诉你的信息。",
    checkpointer=memory,
)

config = {"configurable": {"thread_id": "user-001"}}   # 同一会话用同一 thread_id

# 第 1 轮：告诉它一些信息
r1 = chat_agent.invoke({"messages": [{"role": "user", "content": "你好，我是老王，我的订单号是 SO20260815"}]}, config)
print("Agent:", r1["messages"][-1].content)
print()

# 第 2 轮：不重复提供任何信息，看它是否记得
r2 = chat_agent.invoke({"messages": [{"role": "user", "content": "我的订单到哪了？"}]}, config)
print("Agent:", r2["messages"][-1].content)

In [ ]:
# 换一个 thread_id 就是另一个"人"的会话，互相隔离
config_b = {"configurable": {"thread_id": "user-002"}}
r3 = chat_agent.invoke({"messages": [{"role": "user", "content": "我的订单到哪了？"}]}, config_b)
print("新会话 Agent:", r3["messages"][-1].content)   # 它不认识这个用户

# 原会话的记忆仍在
r4 = chat_agent.invoke({"messages": [{"role": "user", "content": "我刚说我叫什么？"}]}, config)
print("原会话 Agent:", r4["messages"][-1].content)

> `InMemorySaver` 存在内存里，进程重启即丢失。生产环境可换用 `SqliteSaver`、`PostgresSaver` 等持久化实现（`langgraph-checkpoint-*` 系列包），**业务代码一行都不用改**——这就是 checkpointer 抽象的好处。

## 4. response_format：结构化收尾

让 Agent 在完成任务后，把最终结论整理成 Pydantic 对象。放在 `result["structured_response"]` 里。

In [ ]:
from pydantic import BaseModel, Field

class SupportReport(BaseModel):
    answer: str = Field(description="给用户的最终答复")
    tools_used: list[str] = Field(description="解决问题过程中用到的工具名")
    need_human: bool = Field(description="是否需要转人工")

report_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是客服助手，完成任务后按 schema 输出报告。",
    response_format=SupportReport,
)

res = report_agent.invoke({"messages": [{"role": "user", "content": "订单 SO20260901 大概还要多久？另外帮我算下 4999 打 8 折多少"}]})
print(res["structured_response"])

## 5. 链还是 Agent？选型速查

| 场景 | 选型 |
|---|---|
| 固定格式的翻译/摘要/分类 | 链（prompt \| llm \| parser） |
| 输入→固定几步加工→输出 | 链 + RunnableParallel/Lambda |
| 用户意图不定、可能要调工具、多轮交互 | Agent |
| 检索文档回答固定问题 | RAG 链（第 7 章） |
| 复杂多角色协作、审批流 | LangGraph（Agent 的进阶框架） |

经验法则：**先用最简单的链；只有当"步骤无法提前写死、需要模型临场决定"时才上 Agent。**

---

## 练习

**练习 1：图书管理 Agent**
定义 3 个工具：`add_book(title, author)`、`list_books()`、`search_books(keyword)`（用模块级 dict 当"数据库"）。创建带记忆的 Agent，测试两轮会话：
- 第 1 轮："把《三体》加进书架，作者是刘慈欣"
- 第 2 轮："书架上现在有什么？"（观察 Agent 调用 `list_books`）

**练习 2：会算账的客服**
组合 `calculator` + `query_order`，创建 Agent。问它："订单 SO20260901 的商品 4999 元，如果用满 4000 减 500 的券再打 9 折，多少钱？"——观察它先查订单、再分步计算的轨迹。

**练习 3：结构化报告（选做）**
给练习 2 的 Agent 加 `response_format`，让它输出 `{answer, tools_used, confidence}` 三字段报告。

> 提示：工具函数操作模块级 dict 即可，函数内不要用 `global` 也可以直接修改（只增改 key 的话）。

---

## 参考答案（先自己写再看！）

In [ ]:
# 练习 1 参考答案：图书管理 Agent
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

bookshelf = {}   # 模拟数据库：{书名: 作者}

@tool
def add_book(title: str, author: str) -> str:
    """把一本书加入书架。参数：书名、作者。"""
    bookshelf[title] = author
    return f"已添加《{title}》（{author}）"

@tool
def list_books() -> str:
    """列出书架上所有的书。"""
    if not bookshelf:
        return "书架是空的"
    return "\n".join(f"- 《{t}》{a}" for t, a in bookshelf.items())

@tool
def search_books(keyword: str) -> str:
    """按关键词模糊搜索书名或作者。"""
    hits = [f"《{t}》{a}" for t, a in bookshelf.items() if keyword in t or keyword in a]
    return "\n".join(hits) if hits else f"没有找到与 '{keyword}' 相关的书"

lib_agent = create_agent(
    model=llm,
    tools=[add_book, list_books, search_books],
    system_prompt="你是图书管理助手，帮用户维护书架。",
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "lib-1"}}

print(lib_agent.invoke({"messages": [{"role": "user", "content": "把《三体》加进书架，作者刘慈欣；再把《活着》加上，作者余华"}]}, cfg)["messages"][-1].content)
print()
print(lib_agent.invoke({"messages": [{"role": "user", "content": "书架上现在有什么？"}]}, cfg)["messages"][-1].content)

In [ ]:
# 练习 2 参考答案：会算账的客服（观察工具轨迹）
from langchain.agents import create_agent

bill_agent = create_agent(
    model=llm,
    tools=[calculator, query_order],
    system_prompt="你是客服助手。涉及金额计算必须调用计算器工具，分步算清楚。",
)
res = bill_agent.invoke({"messages": [{"role": "user", "content": (
    "订单 SO20260901 的商品 4999 元，先用满 4000 减 500 的优惠券，再打 9 折，最终多少钱？"
)}]})

for m in res["messages"]:
    name = type(m).__name__
    if name == "AIMessage" and m.tool_calls:
        print(f"AI 调用: {[tc['name'] for tc in m.tool_calls]} {[tc['args'] for tc in m.tool_calls]}")
    elif name == "ToolMessage":
        print(f"工具返回: {m.content}")
    elif name == "AIMessage":
        print("最终回答:", m.content)

In [ ]:
# 练习 3 参考答案：结构化报告
from pydantic import BaseModel, Field

class BillReport(BaseModel):
    answer: str = Field(description="最终计算结果与解释")
    tools_used: list[str] = Field(description="用到的工具名列表")
    confidence: float = Field(description="对答案的把握 0-1")

bill_agent2 = create_agent(
    model=llm,
    tools=[calculator, query_order],
    system_prompt="你是客服助手，完成计算后按 schema 汇报。",
    response_format=BillReport,
)
res = bill_agent2.invoke({"messages": [{"role": "user", "content": "4999 先减 500 再打 9 折是多少？"}]})
print(res["structured_response"])